In [16]:
import os,glob,random,time
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score,precision_recall_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
from torchvision.models import ResNet50_Weights

In [17]:
root_dir = "./datasets/MVTecAD"
img_size = 256
device  = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
categories = ["bottle","cable","capsule","carpet","grid","hazelnut","leather","metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]

img_size = 448 
# To ensure dx32x32 output similar to previous models, DINOv2(ViT-B/14) will need (32x14)x(32x14) sized image

blocks = [5,8]
# DINOv2(ViT-B/14) has 12 sequential blocks, Block 5 and 8 are good layers to access for features

SIGMA = 4

# Data Loading

In [18]:
class MVTecDataset(Dataset):
    def __init__(self,root_dir,category,is_train=True,transform=None,mask_transform=None):
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.defect_types = []
        self.transform = transform
        self.mask_transform = mask_transform
        
        if is_train:
            paths = sorted(glob.glob(os.path.join(root_dir,category,"train","good","*.png")))
            self.image_paths.extend(paths)
            self.labels.extend([0]*len(paths))
            self.mask_paths.extend([None]*len(paths))
            self.defect_types.extend(["good"*len(paths)])
        
        else:
            test_dir = os.path.join(root_dir,category,"test")
            for defect in os.listdir(test_dir):
                if defect.startswith('.'):
                    continue
                paths = sorted(glob.glob(os.path.join(test_dir,defect,"*.png")))
                self.image_paths.extend(paths)
                self.defect_types.extend([defect]*len(paths))

                if defect == "good":
                    self.labels.extend([0]*len(paths))
                    self.mask_paths.extend([None]*len(paths))
                else:
                    self.labels.extend([1]*len(paths))
                    for p in paths:
                        self.mask_paths.append(os.path.join(root_dir,category,"ground_truth",defect,os.path.basename(p).replace(".png","_mask.png")))
    
    def __len__(self):
            return len(self.image_paths)
    
    def __getitem__(self,idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        mask_path = self.mask_paths[idx]
        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
            mask = self.mask_transform(mask) if self.mask_transform else transforms.ToTensor()(mask)
        else:
            mask = torch.zeros((1,image.shape[1],image.shape[2]))
        return image,self.labels[idx],mask

In [19]:
def make_transforms(img_size):
    img_trfm = transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
        ])

    mask_trfm = transforms.Compose([
        transforms.Resize((img_size,img_size),interpolation=transforms.InterpolationMode.NEAREST),
        transforms.ToTensor()
    ])
    
    return img_trfm,mask_trfm

IMG_TRFM,MASK_TRFM = make_transforms(img_size)

# ViT Feature Extractor
Returns all patch tokens in (B,C,H,W) shape.

2 blocks of 768 channels concatenated to 1536 channels finally

In [ ]:
class ViTFeatureExtractor(nn.Module):
    def __init__(self,arch="dinov2_vitb14",blocks=(5,8)):
        super().__init__()
        
        self.backbone = torch.hub.load("facebookresearch/dinov2",arch)
        self.blocks = list(blocks)
        
        for param in self.parameters():
            param.requires_grad = False
        self.eval()
    
    def train(self,mode=True):
        return super().train(False)

    @torch.no_grad()
    def forward(self,x):
        feats = self.backbone.get_intermediate_layers(x,n=self.blocks,reshape=True) # returns feats in (B,C,H,W) form
        return torch.cat(feats,dim=1)

    
@torch.no_grad()
def extract_features(dataset,extractor,batch_size=16):
    loader = DataLoader(dataset,batch_size=batch_size,shuffle=False,num_workers=0)
    feats,labels,masks = [],[],[]
    
    for x,y,m in loader:
        feats.append(extractor(x.to(device)).half().cpu())
        labels.append(y)
        masks.append((m>0.5).to(torch.uint8))
    return torch.cat(feats),torch.cat(labels),torch.cat(masks)

# Heads
Same as AE_PaDiM_Ensemble model

In [21]:
class FeatureNormaliser:
    def fit(self,feats):
        n,s,ss = 0,0,0
        for i in range(0,len(feats),16):
            x = feats[i:min(len(feats),i+16)]
            s += x.sum(dim=(0,2,3),dtype=torch.float64)
            ss += (x**2).sum(dim=(0,2,3),dtype=torch.float64)
            n += x.shape[0]*x.shape[2]*x.shape[3]
        mean = s/n
        std = (ss/n - mean**2).clamp_min(0).sqrt()
        
        self.mean = mean.float().view(1,-1,1,1) # Per channel mean
        self.std = std.float().clamp_min(1e-2).view(1,-1,1,1) # Per channel std
        return self

    def __call__(self, x):
        return (x - self.mean.to(x.device))/self.std.to(x.device)

In [22]:
def positional_encoding(H,W,n_freq=4):
    ys,xs = torch.linspace(0,1,H),torch.linspace(0,1,W)
    gy,gx = torch.meshgrid(ys,xs,indexing="ij")
    chans = []
    
    for k in range(n_freq):
        f = (2**k)*np.pi #pi,2pi,4pi,8pi,...
        for g in (gx,gy):
            chans+=[torch.sin(f*g),torch.cos(f*g)]
    return torch.stack(chans)            

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

In [23]:
def conv1x1_bn_relu(i,o):
    return [nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(inplace=True)]

class ConvAutoEncoder(nn.Module):
    def __init__(self,feat=1536,pos=0,hidden=512,bottleneck=128):
        super().__init__()
        self.pos = pos
        self.encoder = nn.Sequential(
            *conv1x1_bn_relu(feat+pos,hidden),*conv1x1_bn_relu(hidden,bottleneck)
        )
        self.decoder = nn.Sequential(
            *conv1x1_bn_relu(bottleneck+pos,hidden),nn.Conv2d(in_channels=hidden,out_channels=feat,kernel_size=1,stride=1)
        )
        
    def forward(self,x,positional=None):
        if self.pos:
            p = positional.unsqueeze(0).expand(x.shape[0],-1,-1,-1)
            z = self.encoder(torch.cat([x,p],1))
            return self.decoder(torch.cat([z,p],1))
        return self.decoder(self.encoder(x))
    
class AEhead:
    def __init__(self,use_pos =False,epochs=60,lr=1e-3,noise_std=0.1,batch_size=32,seed=42,verbose=False):
        self.use_pos,self.epochs,self.lr,self.noise_std,self.batch_size,self.seed,self.verbose = use_pos,epochs,lr,noise_std,batch_size,seed,verbose
        self.name = "ae_pos" if use_pos else "ae"
        
    def fit(self,train_feats):
        set_seed(self.seed)
        N,C,H,W = train_feats.shape
        
        self.norm = FeatureNormaliser().fit(train_feats)
        self.pos = positional_encoding(H,W).to(device) if self.use_pos else None
        self.model = ConvAutoEncoder(C,0 if (self.use_pos == False) else self.pos.shape[0]).to(device)
        
        optimizer = torch.optim.Adam(self.model.parameters(),lr=self.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=self.epochs)
        
        loader = DataLoader(train_feats,batch_size=self.batch_size,shuffle=True,drop_last=len(train_feats)>self.batch_size)
        
        for epoch in range(self.epochs):
            self.model.train()
            total_loss = 0
            for X in loader:
                X = self.norm(X.to(device).float())
                X_in = X + self.noise_std*torch.randn_like(X)
                loss = F.mse_loss(self.model(X_in,self.pos),X)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            scheduler.step()
            if self.verbose and (epoch+1)%10 == 0:
                print(f"[{self.name}] Epoch {epoch + 1}/{self.epochs} : Loss {total_loss/len(loader):.4f}")
        return self

    @torch.no_grad()
    def score_map(self,feats):
        self.model.eval()
        x = self.norm(feats.to(device).float())
        
        return ((x-self.model(x,self.pos))**2).mean(1,keepdim=True).cpu()

In [24]:
class GaussianHead:
    name = "padim"
    def __init__(self,d=200,eps=0.01,seed=42):
        self.d,self.eps,self.seed = d,eps,seed
    
    def fit(self,train_feats):
        N,C,H,W = train_feats.shape # H W N C
        self.indexes = torch.randperm(C,generator=torch.Generator().manual_seed(self.seed))[:self.d]
        self.norm = FeatureNormaliser().fit(train_feats)
        
        m,sd = self.norm.mean[:,self.indexes],self.norm.std[:,self.indexes]
        X = (train_feats[:,self.indexes].float()-m)/sd
        X = X.permute(2,3,0,1).reshape(H*W,N,self.d)
        
        self.mean = X.mean(1)
        
        inv = torch.empty(H*W,self.d,self.d)
        I = torch.eye(self.d,dtype=torch.float64)
        
        for s in range(0,H*W,128):
            Xc = (X[s:s+128]-self.mean[s:s+128,None,:]).double()
            cov = torch.einsum("pnd,pne->pde",Xc,Xc)/max(N-1,1) + self.eps*I
            inv[s:s+128] = torch.linalg.inv(cov).float()
        self.inv,self.H,self.W = inv,H,W
        return self    
    
    @torch.no_grad()
    def score_map(self,feats):
        B = feats.shape[0]
        m,sd = self.norm.mean[:,self.indexes], self.norm.std[:,self.indexes]
        x = (feats[:,self.indexes].float()-m)/sd               
        x = x.permute(2,3,0,1).reshape(self.H*self.W,B,self.d) - self.mean[:,None,:]
        m = (torch.einsum("pbd,pde->pbe",x,self.inv)*x).sum(-1)    
        return m.clamp_min(0).sqrt().reshape(self.H,self.W,B).permute(2,0,1).unsqueeze(1)

In [ ]:
class EnsembleHead:
    name = "ensemble"

    def __init__(self,head_a,head_b):
        self.a,self.b = head_a,head_b

    def fit(self,train_feats):
        self.a.fit(train_feats)
        self.b.fit(train_feats)
        za = self.a.score_map(train_feats).flatten(1)
        zb = self.b.score_map(train_feats).flatten(1)
        self.mu_a,self.sd_a = za.mean(),za.std().clamp_min(1e-6)
        self.mu_b,self.sd_b = zb.mean(),zb.std().clamp_min(1e-6)
        return self

    def score_map(self,feats):
        za = (self.a.score_map(feats)-self.mu_a)/self.sd_a
        zb = (self.b.score_map(feats)-self.mu_b)/self.sd_b
        return (za+zb)/2

# Evaluation

In [26]:
def evaluate(head, test_feats, labels, masks, defects, img_size, sigma=None):
    sigma = SIGMA if sigma is None else sigma
    maps = []
    for i in range(0, len(test_feats), 16):
        m = head.score_map(test_feats[i:i + 16])
        m = F.interpolate(m, size=(img_size, img_size), mode="bilinear", align_corners=False)
        maps.append(m.squeeze(1).numpy())
    amap = np.concatenate(maps)
    if sigma > 0:
        amap = np.stack([gaussian_filter(a, sigma=sigma) for a in amap])

    img_scores = amap.reshape(len(amap), -1).max(1)
    img_auroc = roc_auc_score(labels, img_scores)
    pxl_auroc = roc_auc_score(masks.numpy().reshape(-1), amap.reshape(-1))
    p, r, _ = precision_recall_curve(labels, img_scores)
    best_f1 = float((2 * p * r / (p + r + 1e-12)).max())

    defects = np.asarray(defects)
    per_defect = {d: roc_auc_score(labels[(defects == d) | (defects == "good")], img_scores[(defects == d) | (defects == "good")])
                  for d in sorted(set(defects) - {"good"})}
    return dict(img_auroc=img_auroc, pxl_auroc=pxl_auroc, f1=best_f1, per_defect=per_defect), amap


def run_category(category, extractor, img_size, seeds=(0,), heads=("ae", "ae_pos", "padim", "ensemble"), verbose=True):
    img_tf, mask_tf = make_transforms(img_size)
    train_ds = MVTecDataset(root_dir, category, True, img_tf)
    test_ds = MVTecDataset(root_dir, category, False, img_tf, mask_tf)

    t0 = time.time()
    train_feats, _, _ = extract_features(train_ds, extractor)
    test_feats, labels, masks = extract_features(test_ds, extractor)
    if verbose:
        print(f"[{category}] {len(train_ds)} train / {len(test_ds)} test, features {tuple(train_feats.shape[1:])}, "
              f"extracted in {time.time() - t0:.0f}s")

    make = {"ae":       lambda s: AEhead(use_pos=False, seed=s),
            "ae_pos":   lambda s: AEhead(use_pos=True, seed=s),
            "padim":    lambda s: GaussianHead(seed=s),
            "ensemble": lambda s: EnsembleHead(AEhead(use_pos=True, seed=s), GaussianHead(seed=s))}
    rows = []
    for name in heads:
        for seed in seeds:
            head = make[name](seed).fit(train_feats)
            res, amap = evaluate(head, test_feats, labels, masks, test_ds.defect_types, img_size)
            rows.append(dict(category=category, head=name, seed=seed, **{k: res[k] for k in ("img_auroc", "pxl_auroc", "f1")},
                             per_defect=res["per_defect"]))
            if verbose:
                print(f"  {name:9s} seed {seed}  image AUROC {res['img_auroc']:.4f} | pixel AUROC {res['pxl_auroc']:.4f} | F1 {res['f1']:.3f}")
    return rows

In [27]:
extractor = ViTFeatureExtractor(blocks=blocks).to(device)

SEEDS = (0,)
all_rows = []
for cat in categories:
    print("\n" + "-" * 60)
    all_rows += run_category(cat,extractor,img_size,seeds=SEEDS)

df_vit = pd.DataFrame(all_rows)
df_vit.drop(columns="per_defect").to_csv("results_vit.csv", index=False)

Using cache found in /Users/adityapatil/.cache/torch/hub/facebookresearch_dinov2_main



------------------------------------------------------------
[bottle] 209 train / 83 test, features (1536, 32, 32), extracted in 27s
  ae        seed 0  image AUROC 0.9992 | pixel AUROC 0.9895 | F1 0.992
  ae_pos    seed 0  image AUROC 0.9992 | pixel AUROC 0.9892 | F1 0.992
  padim     seed 0  image AUROC 1.0000 | pixel AUROC 0.9897 | F1 1.000
  ensemble  seed 0  image AUROC 1.0000 | pixel AUROC 0.9904 | F1 1.000

------------------------------------------------------------
[cable] 224 train / 150 test, features (1536, 32, 32), extracted in 39s
  ae        seed 0  image AUROC 0.9239 | pixel AUROC 0.9676 | F1 0.920
  ae_pos    seed 0  image AUROC 0.9278 | pixel AUROC 0.9686 | F1 0.920
  padim     seed 0  image AUROC 0.8731 | pixel AUROC 0.9792 | F1 0.843
  ensemble  seed 0  image AUROC 0.8919 | pixel AUROC 0.9805 | F1 0.859

------------------------------------------------------------
[capsule] 219 train / 132 test, features (1536, 32, 32), extracted in 36s
  ae        seed 0  image AU

In [28]:
cnn = pd.read_csv("results_AE_PaDiM_Ensemble.csv")
cnn["backbone"], df_vit["backbone"] = "cnn", "vit"
both = pd.concat([cnn, df_vit[["category", "head", "seed", "img_auroc", "pxl_auroc", "f1", "backbone"]]])

pivot = both.groupby(["category", "backbone"])["img_auroc"].mean().unstack()
pivot["delta (vit - cnn)"] = pivot["vit"] - pivot["cnn"]
pivot.loc["MEAN"] = pivot.mean()
print(pivot.round(4).to_string())

backbone       cnn     vit  delta (vit - cnn)
category                                     
bottle      1.0000  0.9996            -0.0004
cable       0.8981  0.9042             0.0061
capsule     0.8216  0.9333             0.1117
carpet      0.9421  0.9994             0.0573
grid        0.6861  0.9992             0.3131
hazelnut    0.9754  0.8139            -0.1615
leather     1.0000  0.9981            -0.0019
metal_nut   0.9939  0.9999             0.0060
pill        0.9162  0.9595             0.0433
screw       0.7431  0.8968             0.1536
tile        0.9932  0.9986             0.0053
toothbrush  0.8361  0.9958             0.1597
transistor  0.8807  0.9675             0.0868
wood        0.9836  0.9901             0.0066
zipper      0.9523  0.9924             0.0401
MEAN        0.9082  0.9632             0.0551
